# ⚙️ Arranque de la sesión (todo en uno)

Todo lo necesario tras **abrir o reconectar** el notebook: GPU, Drive, dependencias, variables de entorno, claves y `sys.path`. Ejecuta de arriba abajo y sigue directo al gimnasio — ya no hace falta pasar por `instalacion_colab.ipynb` cada vez (ese queda solo para la **primera** instalación: crear carpetas en Drive, subir el zip y dar de alta los secretos).

**Secretos de Colab** (icono 🔑): `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `VOYAGE_API_KEY`, y opcionales `HF_TOKEN`, `COHERE_API_KEY`.

In [ ]:
# A1 · GPU (opcional pero recomendado: Runtime → Change runtime type → L4)
import torch
if torch.cuda.is_available():
    print("✓ GPU:", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
else:
    print("✗ SIN GPU: los modelos locales irán a CPU (lento pero funciona).")

In [ ]:
# A2 · Montar Google Drive
import os
from google.colab import drive

drive.mount("/content/drive", force_remount=True)
RAG_LAB_DRIVE = "/content/drive/MyDrive/rag_lab"
assert os.path.exists(RAG_LAB_DRIVE), (
    f"No existe {RAG_LAB_DRIVE}: crea la carpeta rag_lab en Drive "
    "(véase instalacion_colab.ipynb, sección 'Antes de empezar').")
print("✓ Drive montado. Contenido:", os.listdir(RAG_LAB_DRIVE))

In [ ]:
# A3 · (Solo si subiste un ZIP nuevo) re-extraer el proyecto
# OJO: extraer PISA el código de rag_lab_proyecto/ con el contenido del zip.
# Déjalo en False para el día a día; ponlo en True tras subir un zip actualizado.
REEXTRAER_ZIP = False

import zipfile, os
zip_path = f"{RAG_LAB_DRIVE}/rag_lab_proyecto.zip"
proyecto = f"{RAG_LAB_DRIVE}/rag_lab_proyecto"
if REEXTRAER_ZIP and os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(proyecto)   # el zip contiene rag_lab/, tests/... en su raíz
    print("✓ Proyecto re-extraído del zip. RECUERDA: Runtime → Restart session "
          "si ya habías importado rag_lab en este kernel.")
elif os.path.exists(proyecto):
    print("✓ Proyecto presente (sin re-extraer).")
else:
    raise FileNotFoundError(
        f"Ni proyecto ni zip: sube rag_lab_proyecto.zip a {RAG_LAB_DRIVE} "
        "y pon REEXTRAER_ZIP=True.")

In [ ]:
# A4 · Dependencias (requirements.txt + Voyage explícito por si el zip es viejo)
import subprocess
req = f"{RAG_LAB_DRIVE}/rag_lab_proyecto/requirements.txt"
print("Instalando dependencias (3–5 min la primera vez, segundos después)...")
r = subprocess.run(["pip", "install", "-q", "-r", req], capture_output=True, text=True)
print("✓ requirements.txt instalado" if r.returncode == 0
      else f"⚠ pip devolvió {r.returncode}:\n{(r.stderr or '')[-800:]}")
r2 = subprocess.run(["pip", "install", "-q", "voyageai", "langchain-voyageai"],
                    capture_output=True, text=True)
print("✓ voyageai + langchain-voyageai instalados" if r2.returncode == 0
      else f"⚠ voyage falló: {(r2.stderr or '')[-400:]}")

In [ ]:
# A5 · Variables de entorno (CRÍTICO: antes de importar rag_lab,
# porque CONFIG es una instantánea del entorno en el momento del import)
import os, pathlib

MEM_DIR = f"{RAG_LAB_DRIVE}/rag_memory"
HF_DIR = f"{RAG_LAB_DRIVE}/hf_cache"
pathlib.Path(MEM_DIR).mkdir(parents=True, exist_ok=True)
pathlib.Path(HF_DIR).mkdir(parents=True, exist_ok=True)
os.environ["RAG_MEMORY_DIR"] = MEM_DIR
os.environ["HF_HOME"] = HF_DIR
print(f"RAG_MEMORY_DIR = {MEM_DIR}")
print(f"HF_HOME        = {HF_DIR}")
print("rag_memory:", os.listdir(MEM_DIR) or "(vacío)")

In [ ]:
# A6 · Claves de API desde los secretos de Colab (también antes del import)
import os
from google.colab import userdata

OBLIGATORIAS = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "VOYAGE_API_KEY"]
OPCIONALES = ["HF_TOKEN", "COHERE_API_KEY"]

faltan = []
for clave in OBLIGATORIAS + OPCIONALES:
    try:
        valor = userdata.get(clave)
    except Exception:
        valor = None
    if valor:
        os.environ[clave] = valor
        print(f"✓ {clave:18} cargada ({valor[:8]}...)")
    else:
        (faltan if clave in OBLIGATORIAS else []).append(clave)
        print(f"  {clave:18} no definida"
              + ("" if clave in OPCIONALES else "  ⚠"))

if faltan:
    print(f"\n⚠ Faltan claves: {faltan}. Añádelas en el gestor de secretos (🔑),"
          " concede acceso a este notebook y re-ejecuta esta celda.")

In [ ]:
# A7 · Copia LOCAL del código + sys.path + import de verificación
#
# El código se copia al disco de la VM (/content) y se importa DESDE AHÍ.
# Motivo: importar desde Drive depende de que el montaje FUSE siga vivo; si se
# cae (errno 107 "Transport endpoint is not connected"), cualquier import o
# traceback revienta a mitad de corrida. Con la copia local, los imports son
# inmunes a las caídas de Drive; en Drive quedan solo los DATOS persistentes
# (rag_memory, hf_cache, pdfs), que se tocan poco y en escrituras atómicas.
import shutil, sys

ORIGEN_DRIVE = f"{RAG_LAB_DRIVE}/rag_lab_proyecto"
RAIZ_PROYECTO = "/content/rag_lab_proyecto"          # copia local (disco de la VM)

shutil.copytree(ORIGEN_DRIVE, RAIZ_PROYECTO, dirs_exist_ok=True)
print(f"✓ Código copiado a {RAIZ_PROYECTO} (imports inmunes a caídas de Drive)")

for p in (RAIZ_PROYECTO, f"{RAIZ_PROYECTO}/tests"):
    if p not in sys.path:
        sys.path.insert(0, p)

from rag_lab import CONFIG, MetaSupervisor, Plan
from rag_lab.memory import Memoria
avisos = CONFIG.validar() if hasattr(CONFIG, "validar") else []
print("✓ rag_lab importado" + (f" | avisos: {avisos}" if avisos else ""))
print("Memoria en:", CONFIG.fichero_memoria)

# Verificación defensiva: si el import no viene de la copia local, ABORTAR con
# mensaje claro (tocaya 'rag_lab' en el path, extracción rota, arranque viejo).
import rag_lab
_origen = getattr(rag_lab, "__file__", None)
assert _origen and _origen.startswith(RAIZ_PROYECTO), \
    f"rag_lab importado desde {_origen!r}: reextrae el zip y reinicia el kernel"
print(f"✓ rag_lab importado desde la copia local: {_origen}")

---
# 🏋️ Y ahora, el gimnasio
Las celdas siguientes son el flujo de entrenamiento de siempre. La primera repite parte del arranque (claves comentadas, `sys.path`): es **idempotente**, así que no estorba — y permite seguir usándola sola en local, donde el arranque de Colab de arriba no aplica.

# 🏋️ Entrenar — el gimnasio del laboratorio
Un documento con plan explicito, tandas resumibles sobre el corpus, y el parte del ultimo experimento.


In [ ]:
import sys, os, json, pathlib, re
import numpy as np
import matplotlib.pyplot as plt

# 1) API keys ANTES de importar rag_lab (CONFIG es una instantanea al importar)
# os.environ['OPENAI_API_KEY'] = '...'
# os.environ['ANTHROPIC_API_KEY'] = '...'
# os.environ['VOYAGE_API_KEY'] = '...'   # opcional
# os.environ['COHERE_API_KEY'] = '...'   # opcional

# 2) Ruta del proyecto: HARDCODEADA (en local, cambia esta linea a tu ruta)
# RAIZ para DATOS (pdfs) sigue en Drive; el CODIGO se importa de la
# copia local /content/rag_lab_proyecto si existe (ver arranque A7).
RAIZ = pathlib.Path('/content/drive/MyDrive/rag_lab/rag_lab_proyecto')
_local = pathlib.Path('/content/rag_lab_proyecto')
sys.path.insert(0, str(_local if _local.exists() else RAIZ))
sys.path.insert(0, str(RAIZ))

from rag_lab.config import CONFIG
from rag_lab.memory import Memoria
mem = Memoria(CONFIG.fichero_memoria)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25})
print('Memoria:', CONFIG.fichero_memoria)
print('Resumen:', mem.resumen())


## 1. Verificacion rapida de despliegue
Tres comprobaciones sobre el codigo REALMENTE cargado. Si algo sale ✗: zip viejo o kernel sin reiniciar.


In [ ]:
from rag_lab.supervisor.meta import MetaSupervisor
print('[✓] perfil determinista' if (RAIZ/'rag_lab/perfil.py').exists() else '[✗] falta perfil.py')
print('[✓] memoria v2 (priors)' if hasattr(Memoria, 'priors') else '[✗] memoria vieja')
print('[✓] grafo sin nodo proyectar' if not hasattr(MetaSupervisor, '_nodo_proyectar_embeddings') else '[✗] meta.py viejo')


## 2. Un documento, un plan explicito
Los tres bancos habituales, ceteris paribus, con el presupuesto reactivo (runs 3→5, preguntas 10→40).


In [ ]:
from rag_lab.supervisor.plan import Plan
meta = MetaSupervisor(mem)
PDF = RAIZ.parent / 'pdfs' / 'MI_DOCUMENTO.pdf'   # <-- ajusta
plan = Plan(bancos=['splitter', 'embedding', 'retriever'], n_runs=3,
            presupuesto={'max_runs': 5, 'max_preguntas': 40}, ceteris_paribus=True)
resultado = meta.ejecutar(str(PDF), plan=plan)


## 3. Tanda resumible sobre el corpus
Recorre `pdfs/`, salta lo ya estudiado (hash en memoria) y sobrevive a cortes de Colab.


In [ ]:
from rag_lab.corpus import hash_pdf
pdfs = sorted((RAIZ.parent / 'pdfs').glob('*.pdf'))
print(len(pdfs), 'PDFs en el corpus')
for i, pdf in enumerate(pdfs, 1):
    if mem.documento_estudiado(doc_hash=hash_pdf(str(pdf))):
        print(f'[{i}/{len(pdfs)}] ya estudiado: {pdf.name}'); continue
    print(f'\n[{i}/{len(pdfs)}] {pdf.name}')
    try:
        meta.ejecutar(str(pdf), plan=plan)
    except Exception as e:
        print('  fallo (sigo con el siguiente):', e)


## 4. El parte del ultimo experimento
Lo esencial de la ultima decision registrada: certezas, stack coronado, Fase B y priors.


In [ ]:
d = mem._leer()
ses = d['sesiones'][-1]; rid = ses['run_id']
stacks = [x['stack'] for x in d['decisiones'] if x['run_id'] == rid]
stack = stacks[-1] if stacks else {}
print('doc     :', ses.get('pdf_nombre'))
print('perfil  :', ses.get('perfil'))
print('\nCERTEZAS:')
for b, c in (stack.get('certezas') or {}).items():
    ic = c.get('ic_primaria') or {}
    print(f"  {b:10} ganador={c.get('ganador')} resuelto={c.get('resuelto')} "
          f"acuerdo={c.get('acuerdo')} IC_ampl={ic.get('amplitud')} motivo={c.get('motivo')}")
print('\nSTACK:')
for b, v in (stack.get('stack_recomendado') or {}).items():
    print(f"  {b:10} {v.get('etiqueta')} (via={v.get('via', 'sintesis')})")
print('\nECO :', {b: v.get('etiqueta') for b, v in (stack.get('stack_alternativo_economico') or {}).items()})
print('FASE B:', stack.get('fase_b') or 'sin empates que dirimir')
print('confianza:', stack.get('confianza_global'))
print('tasa acierto priors:', mem.tasa_acierto() or 'sin predicciones resueltas aun')
